In [ ]:
import matplotlib.pyplot as plt
import cupy as cp

from scripts.constants import *
from scripts.hamiltonian import *
from scripts.self_consistency import *


In [ ]:
def free_energy_non_sc(H, U=0, V=0, V_prime=0, Delta0=0, 
                       Delta_s=0,  Delta_d=0,  Delta_px=0,  Delta_py=0,
                       Delta_uu_px=0, Delta_uu_py=0, 
                       Delta_dd_px=0, Delta_dd_py=0):
    temperature = H.temperature
    eps = cp.linalg.eigvalsh(H.matrix)
    eps = eps[eps > 0]
    E_S = 0
    E_S -= (cp.abs(Delta0)**2)/U if not U==0 else 0
    E_S -= (cp.abs(Delta_s)**2)/V if not V==0 else 0
    E_S -= (cp.abs(Delta_d)**2)/V if not V==0 else 0
    E_S -= (cp.abs(Delta_px)**2)/V if not V==0 else 0
    E_S -= (cp.abs(Delta_py)**2)/V if not V==0 else 0

    E_S -= (cp.abs(Delta_dd_py)**2)/V_prime if not V_prime==0 else 0
    E_S -= (cp.abs(Delta_dd_py)**2)/V_prime if not V_prime==0 else 0
    E_S -= (cp.abs(Delta_uu_py)**2)/V_prime if not V_prime==0 else 0
    E_S -= (cp.abs(Delta_uu_py)**2)/V_prime if not V_prime==0 else 0

    internal_energy = -(1 / 2) * cp.sum(eps)
    if temperature == 0:
        S = 0
    elif temperature > 0:
        S = cp.sum(cp.log(1 + cp.exp(-eps / temperature)))

    F = internal_energy - temperature * S + E_S

    return F

In [ ]:
t = 1
mu = -1.8* t
Y,X = 25,25

lattice = Lattice(X, Y, pbc_x=True, pbc_y=True)

U0 = 2.0 * t 
U = U0 * cp.ones(lattice.num_sites)
V0 = 2.0 * t 
V = V0 * cp.ones(lattice.num_sites)

gap_0 = 0.0 * t
gap_p= cp.array([0.1, 0.1j]) * t
gap_d = 0.0 *  t

H = Hamiltonian(lattice, temperature=0)
block = cp.zeros((4, 4), dtype=cp.complex128)

for i in range(lattice.num_sites):
    block[:,:]=0
    block[:2, :2] = -mu * s0
    block[2:, 2:] = mu * s0
    block[:2, 2:] = -1j * gap_0 * s2
    block[2:, :2] = (-1j * gap_0 * s2).conj().T
    H.set_block(i,i, block)

for i, j in lattice.edges:
    block[:,:]=0
    block[0:2, 0:2] = -t * s0
    block[2:4, 2:4] = t * s0

    gap_ij = unconventional_gap(lattice.get_disp(i,j), gap_p=gap_p)
    block[0:2, 2:4] = gap_ij
    gap_ji = unconventional_gap(lattice.get_disp(j,i), gap_p=gap_p)
    block[2:4, 0:2] = gap_ji.conj().T

    H.set_block(i,j, block)

print(free_energy_non_sc(H,0,0,0,Delta_px=0, Delta_py=0))
print(H.free_energy(0,0,0,False))

In [ ]:
delta0_list = 2 * cp.arange(-10 * 0.0408, 5 * 0.0408, 0.0204, dtype=cp.complex128)
factors = cp.array([1, 1j],dtype=cp.complex128)
t = 1
mu = -1.8* t
Y,X = 25,25
lattice = Lattice(X, Y, pbc_x=True, pbc_y=True)
free_energies = []
for dx in delta0_list:
    for factor in factors:
        H = Hamiltonian(lattice, temperature=0)

        gap_p = cp.array([0, factor * dx], dtype=cp.complex128)

        block = cp.zeros((4, 4), dtype=cp.complex128)

        for i in range(lattice.num_sites):
            block[:,:]=0
            block[:2, :2] = -mu * s0
            block[2:, 2:] = mu * s0
            H.set_block(i,i, block)

        for i, j in lattice.edges:
            block[:,:]=0
            block[0:2, 0:2] = -t * s0
            block[2:4, 2:4] = t * s0

            gap_ij = unconventional_gap(lattice.get_disp(i,j), gap_p=gap_p)
            block[0:2, 2:4] = gap_ij
            gap_ji = unconventional_gap(lattice.get_disp(j,i), gap_p=gap_p)
            block[2:4, 0:2] = gap_ji.conj().T

            H.set_block(i,j, block)

        temp = free_energy_non_sc(H, U=0, V=2, V_prime=0, Delta_px=dx, Delta_py=factor*dx)
        free_energies.append(temp.get())

In [ ]:
plt.figure()
plt.plot(cp.asnumpy(delta0_list), cp.asnumpy(free_energies)[::2])
plt.plot(cp.asnumpy(delta0_list), cp.asnumpy(free_energies)[1::2])


In [ ]:
dx = cp.complex128(0.0408 + 0j)

angles = cp.linspace(0, 2 * cp.pi, 360)
factors = cp.exp(1j*angles)
t = 1
mu = -1.8* t
Y,X = 25,25
lattice = Lattice(X, Y, pbc_x=True, pbc_y=True)
free_energies = []

for factor in factors:
    H = Hamiltonian(lattice, temperature=0)

    gap_p = cp.empty((2,), dtype=cp.complex128)
    gap_p[0] = 0
    gap_p[1] = factor * dx
    
    block = cp.zeros((4, 4), dtype=cp.complex128)

    for i in range(lattice.num_sites):
        block[:,:]=0
        block[:2, :2] = -mu * s0
        block[2:, 2:] = mu * s0
        H.set_block(i,i, block)

    for i, j in lattice.edges:
        block[:,:]=0
        block[0:2, 0:2] = -t * s0
        block[2:4, 2:4] = t * s0

        gap_ij = unconventional_gap(lattice.get_disp(i,j), gap_p=gap_p)
        block[0:2, 2:4] = gap_ij
        gap_ji = unconventional_gap(lattice.get_disp(j,i), gap_p=gap_p)
        block[2:4, 0:2] = gap_ji.conj().T

        H.set_block(i,j, block)

    temp = free_energy_non_sc(H, U=0, V=2, V_prime=0, Delta_px=dx, Delta_py=factor*dx)
    free_energies.append(temp.get())
    print(temp)